In [282]:
import pandas as pd

ds = pd.read_csv('../../../data/week4_korea_cli.csv')

In [283]:
ds.observation_date = pd.to_datetime(ds.observation_date)
ds = ds.sort_values('observation_date')

In [284]:
ds.isna().sum()

observation_date    0
KORLOLITOAASTSAM    0
dtype: int64

In [285]:
ds.describe()

,observation_date,KORLOLITOAASTSAM
count,438,438.000000
mean,2008-03-16 18:27:56.712328,99.993123
min,1990-01-01 00:00:00,94.805464
25%,1999-02-08 00:00:00,98.979399
50%,2008-03-16 12:00:00,99.983887
75%,2017-04-23 12:00:00,100.942811
max,2026-06-01 00:00:00,105.562636
std,NaN,1.714661


### (아래 코드 셀) 중요!!!

In [315]:
expected_dates = pd.date_range(start=ds.observation_date.min(), end=ds.observation_date.max(), freq='MS')

actual_dates = pd.DatetimeIndex(ds.observation_date)

missing_dates = expected_dates.difference(actual_dates)

missing_dates

DatetimeIndex([], dtype='datetime64[us]', freq='MS')

In [286]:
ds.observation_date.duplicated()

0      False
1      False
2      False
3      False
4      False
       ...  
433    False
434    False
435    False
436    False
437    False
Name: observation_date, Length: 438, dtype: bool

In [287]:
ds.shape

(438, 2)

In [288]:
ds.columns

Index(['observation_date', 'KORLOLITOAASTSAM'], dtype='str')

In [289]:
ds.dtypes

observation_date    datetime64[us]
KORLOLITOAASTSAM           float64
dtype: object

In [290]:
ds.observation_date.is_monotonic_increasing

True

In [291]:
ds.loc[ds.isna().any(axis=1)]

,observation_date,KORLOLITOAASTSAM


In [292]:
ds

,observation_date,KORLOLITOAASTSAM
0,1990-01-01,100.461861
1,1990-02-01,100.462284
2,1990-03-01,100.536374
3,1990-04-01,100.613811
4,1990-05-01,100.646672
...,...,...
433,2026-02-01,101.597894
434,2026-03-01,102.001885
435,2026-04-01,102.363604
436,2026-05-01,102.657914


In [293]:
# Index, Seasonally Adjusted (사이트에서 찾은 정보)

### PART B 시작

In [294]:
ds['target_next_month'] = ds.KORLOLITOAASTSAM.shift(-1)
ds['baseline'] = ds.KORLOLITOAASTSAM
ds['cli_lag1'] = ds.KORLOLITOAASTSAM.shift(1)
ds['cli_rolling3'] = ds.KORLOLITOAASTSAM.rolling(window=3).mean()

In [295]:
ds

,observation_date,KORLOLITOAASTSAM,target_next_month,baseline,cli_lag1,cli_rolling3
0,1990-01-01,100.461861,100.462284,100.461861,NaN,NaN
1,1990-02-01,100.462284,100.536374,100.462284,100.461861,NaN
2,1990-03-01,100.536374,100.613811,100.536374,100.462284,100.486840
3,1990-04-01,100.613811,100.646672,100.613811,100.536374,100.537490
4,1990-05-01,100.646672,100.610137,100.646672,100.613811,100.598952
...,...,...,...,...,...,...
433,2026-02-01,101.597894,102.001885,101.597894,101.190196,101.200525
434,2026-03-01,102.001885,102.363604,102.001885,101.597894,101.596658
435,2026-04-01,102.363604,102.657914,102.363604,102.001885,101.987795
436,2026-05-01,102.657914,102.869808,102.657914,102.363604,102.341134


In [296]:
ds = ds.dropna(subset=['target_next_month', 'cli_lag1', 'cli_rolling3'])

In [297]:
ds

,observation_date,KORLOLITOAASTSAM,target_next_month,baseline,cli_lag1,cli_rolling3
2,1990-03-01,100.536374,100.613811,100.536374,100.462284,100.486840
3,1990-04-01,100.613811,100.646672,100.613811,100.536374,100.537490
4,1990-05-01,100.646672,100.610137,100.646672,100.613811,100.598952
5,1990-06-01,100.610137,100.477904,100.610137,100.646672,100.623540
6,1990-07-01,100.477904,100.254061,100.477904,100.610137,100.578238
...,...,...,...,...,...,...
432,2026-01-01,101.190196,101.597894,101.190196,100.813483,100.830936
433,2026-02-01,101.597894,102.001885,101.597894,101.190196,101.200525
434,2026-03-01,102.001885,102.363604,102.001885,101.597894,101.596658
435,2026-04-01,102.363604,102.657914,102.363604,102.001885,101.987795


In [298]:
ds = ds[(ds.observation_date >= pd.to_datetime('2000-01-01')) & (ds.observation_date <= pd.to_datetime('2024-12-01'))].copy()

In [299]:
ds

,observation_date,KORLOLITOAASTSAM,target_next_month,baseline,cli_lag1,cli_rolling3
120,2000-01-01,104.316848,103.845692,104.316848,104.766040,104.742403
121,2000-02-01,103.845692,103.419138,103.845692,104.316848,104.309527
122,2000-03-01,103.419138,103.058768,103.419138,103.845692,103.860559
123,2000-04-01,103.058768,102.727027,103.058768,103.419138,103.441199
124,2000-05-01,102.727027,102.377781,102.727027,103.058768,103.068311
...,...,...,...,...,...,...
415,2024-08-01,99.558838,99.470364,99.558838,99.631133,99.622570
416,2024-09-01,99.470364,99.382575,99.470364,99.558838,99.553445
417,2024-10-01,99.382575,99.319054,99.382575,99.470364,99.470592
418,2024-11-01,99.319054,99.294928,99.319054,99.382575,99.390664


In [300]:
train = ds[ds.observation_date <= pd.to_datetime('2019-12-01')].copy()
test = ds[(ds.observation_date >= pd.to_datetime('2020-01-01')) & (ds.observation_date <= pd.to_datetime('2024-11-01'))].copy()

In [301]:
train

,observation_date,KORLOLITOAASTSAM,target_next_month,baseline,cli_lag1,cli_rolling3
120,2000-01-01,104.316848,103.845692,104.316848,104.766040,104.742403
121,2000-02-01,103.845692,103.419138,103.845692,104.316848,104.309527
122,2000-03-01,103.419138,103.058768,103.419138,103.845692,103.860559
123,2000-04-01,103.058768,102.727027,103.058768,103.419138,103.441199
124,2000-05-01,102.727027,102.377781,102.727027,103.058768,103.068311
...,...,...,...,...,...,...
355,2019-08-01,98.922328,98.912319,98.922328,98.976546,98.986396
356,2019-09-01,98.912319,98.954759,98.912319,98.922328,98.937064
357,2019-10-01,98.954759,99.036548,98.954759,98.912319,98.929802
358,2019-11-01,99.036548,99.127853,99.036548,98.954759,98.967875


In [302]:
ds.loc[ds.observation_date == pd.to_datetime('2024-03-01')]

,observation_date,KORLOLITOAASTSAM,target_next_month,baseline,cli_lag1,cli_rolling3
410,2024-03-01,99.572229,99.660474,99.572229,99.434278,99.421764


In [303]:
from sklearn.linear_model import LinearRegression
import numpy as np

model = LinearRegression()

In [304]:
#X_train_base = train[['baseline']]

X_train = train[['cli_lag1', 'cli_rolling3']]
y_train = train['target_next_month']

X_test = test[['cli_lag1', 'cli_rolling3']]

In [305]:
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](2,)","[-0.93, 1.83]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](2,)","['cli_lag1','cli_rolling3']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,9.549
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(2)


In [306]:
model_predict = model.predict(X_test)

MAE_base = (test['target_next_month'] - test['baseline']).abs().mean()
MAE_model = (test['target_next_month'] - model_predict).abs().mean()

RMSE_base = np.sqrt(((test['target_next_month'] - test['baseline'])**2).mean())
RMSE_model = np.sqrt(((test['target_next_month'] - model_predict)**2).mean())

#MAE_base, MAE_model, RMSE_base, RMSE_model

In [307]:
result = pd.DataFrame(index=['base', 'model'])

#result.loc['MAE'] = MAE_base

result.loc['base', 'MAE'] = MAE_base
result.loc['model', 'MAE'] = MAE_model

result.loc['base', 'RMSE'] = RMSE_base
result.loc['model', 'RMSE'] = RMSE_model


result

,MAE,RMSE
base,0.194231,0.224456
model,0.363217,0.446527
